In [1]:
!pip install -q faiss-cpu beir sentence-transformers transformers \
    rouge-score sacrebleu bert-score evaluate \
    matplotlib seaborn pandas scikit-learn tqdm scipy --upgrade

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 M

In [2]:
import os, time, random, gc, warnings, math
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, faiss
from tqdm.auto import tqdm

import torch
from sklearn.decomposition import PCA
from sklearn.utils.extmath import randomized_svd
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import normalize
from sklearn.metrics import ndcg_score
from scipy.optimize import curve_fit

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM

from rouge_score import rouge_scorer
from bert_score import score as bert_score

from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_palette("Set2")
pd.options.display.float_format = "{:.4f}".format
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RNG    = np.random.default_rng(42)


2025-05-13 11:08:49.085781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747134529.295944      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747134529.354390      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
DATA_DIR = Path("datasets")
DATASET_URLS = {
    "scifact"     : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
    "scidocs"  : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scidocs.zip",
    "fiqa" : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip",
    "nfcorpus"       : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip"
}


In [4]:
#Add Your Datasets to DATASET_LIST
DATASET_LIST = ["scifact","scidocs", "fiqa", "nfcorpus"]

In [5]:
def load_dataset(name, split="test"):
    url = DATASET_URLS[name]
    path = util.download_and_unzip(url, str(DATA_DIR))
    corpus, queries, qrels = GenericDataLoader(path).load(split=split)
    doc_ids = sorted(corpus.keys()); query_ids = sorted(queries.keys())
    docs   = [corpus[d]["text"] for d in doc_ids]
    qtexts = [queries[q] for q in query_ids]
    relevant=[]
    for qid in query_ids:
        idx=[doc_ids.index(d) for d in qrels.get(qid,{}) if d in doc_ids]
        relevant.append(np.asarray(idx,dtype=int))
    return {"docs":docs,"queries":qtexts,"relevant":relevant}

In [6]:
#Add Your Embedding Models to EMBED_MODELS
EMBED_MODELS = {
    "mpnet"  : "sentence-transformers/all-mpnet-base-v2"
}

embedders = {k: SentenceTransformer(v, device=DEVICE) for k,v in EMBED_MODELS.items()}
precomputed={}
for ds in DATASET_LIST:
    print(f"Embedding {ds}")
    data = load_dataset(ds)
    precomputed[ds] = {"docs":data["docs"],"queries":data["queries"],
                       "relevant":data["relevant"],"embs":{}}
    for eb, mdl in embedders.items():
        d_emb = mdl.encode(data["docs"], batch_size=128, show_progress_bar=False, convert_to_numpy=True)
        q_emb = mdl.encode(data["queries"], batch_size=128, show_progress_bar=False, convert_to_numpy=True)
        
        mu = d_emb.mean(axis = 0, keepdims = True)
        d_emb = d_emb - mu
        q_emb = q_emb - mu
        d_emb = normalize(d_emb); q_emb = normalize(q_emb)
        precomputed[ds]["embs"][eb] = {"doc_emb":d_emb,"qry_emb":q_emb}
        print(f"{eb}: {d_emb.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

  0%|          | 0/57638 [00:00<?, ?it/s]

  mpnet: (57638, 768)
Embedding nfcorpus


datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

  mpnet: (3633, 768)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_topk_cosine(query_emb, doc_emb, k=10):
    scores = cosine_similarity(query_emb, doc_emb)
    return np.argsort(-scores, axis=1)[:, :k]

def faiss_lsh_index(doc_emb, n_bits=None):
    d = doc_emb.shape[1]
    if n_bits is None:
        n_bits = d * 2  # more bits = better accuracy
    index = faiss.IndexLSH(d, n_bits)
    faiss.normalize_L2(doc_emb)
    index.add(doc_emb)
    return index

def retrieve_lsh(index, query_emb, k=10):
    faiss.normalize_L2(query_emb)
    _, I = index.search(query_emb, k)
    return I

def faiss_pq_index(doc_emb, m=16, nbits=8):
    d = doc_emb.shape[1]
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexPQ(d, m, nbits)
    index.train(doc_emb)
    index.add(doc_emb)
    return index

def retrieve_pq(index, query_emb, k=10):
    _, I = index.search(query_emb, k)
    return I

def recall_at_k(pred_indices, ground_truth, k=10):
    correct = 0
    for i in range(len(ground_truth)):
        gt_set = set(ground_truth[i])
        pred_set = set(pred_indices[i][:k])
        correct += len(gt_set & pred_set)
    return correct / (len(ground_truth) * k)

def cosine_distortion(orig_emb, comp_emb, n_samples=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = orig_emb.shape[0]
    idx_i = rng.integers(0, n, size=n_samples)
    idx_j = rng.integers(0, n, size=n_samples)
    orig = normalize(orig_emb)
    comp = normalize(comp_emb)
    cos_orig = np.sum(orig[idx_i] * orig[idx_j], axis=1)
    cos_comp = np.sum(comp[idx_i] * comp[idx_j], axis=1)
    return np.mean(np.abs(cos_orig - cos_comp))

def knn_overlap_at_k(orig_emb, comp_emb, k=10):
    from sklearn.metrics.pairwise import cosine_similarity
    sim_orig = cosine_similarity(orig_emb)
    sim_comp = cosine_similarity(comp_emb)
    overlap = []
    for i in range(sim_orig.shape[0]):
        orig_top = np.argsort(-sim_orig[i])[:k]
        comp_top = np.argsort(-sim_comp[i])[:k]
        overlap.append(len(set(orig_top) & set(comp_top)) / k)
    return np.mean(overlap)

def precision_at_k(pred_indices, gt_indices, k=10):
    precisions = []
    for preds, gts in zip(pred_indices, gt_indices):
        precisions.append(len(set(preds[:k]) & set(gts)) / k)
    return np.mean(precisions)

def mrr_at_k(pred_indices, gt_indices, k=10):
    rr = []
    for preds, gts in zip(pred_indices, gt_indices):
        rank = next((i+1 for i,doc in enumerate(preds[:k]) if doc in gts), None)
        rr.append(1.0/rank if rank else 0.0)
    return np.mean(rr)

In [8]:
def randsvd_Q(M, k, p=10, q=2, device=None):
    if device is None:
        device = M.device

    m, n = M.shape
    l = k + p

    Omega = torch.randn(n, l, device=device) 
    Y = M @ Omega 
    for _ in range(q): 
        Y = M @ (M.T @ Y)

    Q, _ = torch.linalg.qr(Y, mode='reduced')

    return Q

def randsvd_decompose_and_project(docs_np, queries_np, k, p=10, q=2, device='cuda'):
    M = torch.from_numpy(docs_np).to(device)  
    
    Qr = randsvd_Q(M.T, k, p=p, q=q, device=device)
    
    Qr_k = Qr[:, :k]  # (d, k)
    
    docs_rsvd = (M @ Qr_k).cpu().numpy()          
    queries_rsvd = queries_np @ Qr_k.cpu().numpy()  
    
    return docs_rsvd, queries_rsvd

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

device = "cpu"
en_fr_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
en_fr_mdl = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-fr").to(device)
fr_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
fr_en_mdl = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-fr-en").to(device)

def back_translate(text):
    fr_input = en_fr_tok(text, return_tensors="pt", truncation=True, padding=True).to(device)
    fr_out = en_fr_mdl.generate(**fr_input, max_length=128)
    fr_text = en_fr_tok.batch_decode(fr_out, skip_special_tokens=True)[0]
    en_input = fr_en_tok(fr_text, return_tensors="pt", truncation=True, padding=True).to(device)
    en_out = fr_en_mdl.generate(**en_input, max_length=128)
    return fr_en_tok.batch_decode(en_out, skip_special_tokens=True)[0]

query_ds, doc_ds = "scifact", "scidocs"
docs        = precomputed[doc_ds]["embs"]["mpnet"]["doc_emb"]
queries     = precomputed[query_ds]["embs"]["mpnet"]["qry_emb"]
q_texts     = precomputed[query_ds]["queries"]
gt_top      = retrieve_topk_cosine(queries, docs, k=10)


d = docs.shape[1]
lsh_idx = faiss.IndexLSH(d, d)
lsh_idx.add(docs.astype("float32"))

# PQ index
pq_idx = faiss.IndexPQ(d, 64, 8)  # 64 subquantizers, 8 bits each
pq_idx.train(docs.astype("float32"))
pq_idx.add(docs.astype("float32"))

# RandSVD
k_rsvd, p, q_iters = 256, 10, 2
D_t = torch.from_numpy(docs).to(device)
Qr = randsvd_Q(D_t.T, k_rsvd, p=p, q=q_iters, device=device)
Qk = Qr[:, :k_rsvd].cpu().numpy()
docs_rsvd = normalize(docs @ Qk)

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)
n_eval = 200
drops = {"LSH": [], "PQ": [], "RandSVD": []}

for i in range(n_eval):
    orig_q = q_texts[i]
    rel = set(gt_top[i])
    if not rel:
        continue

    bt_q = back_translate(orig_q)
    pe = embedder.encode([bt_q], convert_to_numpy=True)
    pe = normalize(pe)

    #LSH 
    preds_l = retrieve_lsh(lsh_idx, pe, k=10)[0]
    recall_l = len(rel & set(preds_l)) / len(rel)
    drops["LSH"].append(1.0 - recall_l)
    
    #PQ 
    preds_p = retrieve_pq(pq_idx, pe, k=10)[0]
    recall_p = len(rel & set(preds_p)) / len(rel)
    drops["PQ"].append(1.0 - recall_p)

    #RandSVD 
    pe_r = normalize(pe @ Qk)
    preds_r = retrieve_topk_cosine(pe_r, docs_rsvd, k=10)[0]
    recall_r = len(rel & set(preds_r)) / len(rel)
    drops["RandSVD"].append(1.0 - recall_r)

#Summary
for m, vals in drops.items():
    mean_drop = np.mean(vals)
    std_drop  = np.std(vals)
    print(f"{m}: Recall@10 drop = {mean_drop} ± {std_drop}")


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

LSH: Recall@10 drop = 0.48149999999999993 ± 0.18763728307561908
PQ: Recall@10 drop = 0.489 ± 0.19125637244285482
RandSVD: Recall@10 drop = 0.40049999999999997 ± 0.19222317758272545
